# Google Drive의 GK-2A NC를 관측소별 피처로 변환

이 노트북은 Drive의 NC 파일을 재귀적으로 찾고, 공식 96개 관측소의 중심·5km·15km 패치 통계를 추출한 뒤 결과를 Drive에 저장합니다.

- 기본 대상: 2019~2025년 6·7·8월, 14:00 KST
- KMA 파일명은 UTC로 해석하므로 `0500` 파일이 `1400 KST`에 대응합니다.
- 기본적으로 16채널이 모두 모인 날짜만 처리합니다.
- 중단 후 다시 실행하면 완료된 날짜별 결과를 재사용합니다.

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. SME_DATA 최신 코드 설치

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_REF = 'agent/colab-drive-nc-preprocessing'  # PR 병합 후에는 main으로 바꿔도 됩니다.
REPO_DIR = Path('/content/SME_DATA')

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', REPO_REF, f'origin/{REPO_REF}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
print('코드 설치 완료:', REPO_DIR)

## 3. 경로와 기간 설정

`NC_INPUT_DIR`와 `RESULT_DIR`를 본인의 Drive 폴더에 맞게 바꾸세요.

In [ ]:
NC_INPUT_DIR = '/content/drive/MyDrive/SME_DATA/nc'
RESULT_DIR = '/content/drive/MyDrive/SME_DATA/processed_station_features'
STATION_LIST = str(REPO_DIR / 'data/metadata/station_list.csv')

START_DATE = '2019-06-01'
END_DATE = '2025-08-31'
MONTHS = [6, 7, 8]
TARGET_HOUR_KST = 14
FILENAME_TIMEZONE = 'utc'
PATCH_RADII_KM = [0, 5, 15]

# Drive 용량을 아끼려면 False. True면 전체 압축 CSV(.csv.gz)도 생성합니다.
WRITE_CSV = False
# 업로드가 덜 된 날짜는 다음 실행에서 처리하는 것이 안전하므로 기본 False입니다.
ALLOW_PARTIAL = False
# Drive의 HDF5/NetCDF를 Colab 로컬로 하나씩 복사해 읽어 I/O 오류를 줄입니다.
STAGE_DIR = '/content/gk2a_nc_stage'

assert Path(NC_INPUT_DIR).is_dir(), f'입력 폴더가 없습니다: {NC_INPUT_DIR}'
Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)
print('NC 입력:', NC_INPUT_DIR)
print('결과 저장:', RESULT_DIR)

## 4. 실행 명령 구성

In [ ]:
BASE_CMD = [
    sys.executable, str(REPO_DIR / 'scripts/preprocess_drive_nc.py'),
    '--input-dir', NC_INPUT_DIR,
    '--output-dir', RESULT_DIR,
    '--station-list', STATION_LIST,
    '--start-date', START_DATE,
    '--end-date', END_DATE,
    '--months', *[str(value) for value in MONTHS],
    '--target-hour-kst', str(TARGET_HOUR_KST),
    '--filename-timezone', FILENAME_TIMEZONE,
    '--radii-km', *[str(value) for value in PATCH_RADII_KM],
    '--stage-dir', STAGE_DIR,
]
if WRITE_CSV:
    BASE_CMD.append('--write-csv')
if ALLOW_PARTIAL:
    BASE_CMD.append('--allow-partial')

print('명령 준비 완료')

## 5. 먼저 스캔만 실행

NC 파일을 열지 않고 날짜와 채널 구성을 확인합니다. `16채널 완성 시각` 개수를 확인하세요.

In [ ]:
subprocess.run(BASE_CMD + ['--dry-run'], check=True)

## 6. 실제 전처리 실행

처음에는 시험하려면 아래 명령 끝에 `+ ['--limit', '1']`을 붙이세요. 전체 실행은 현재 셀 그대로 실행합니다.

In [ ]:
subprocess.run(BASE_CMD, check=True)

## 7. 결과 검수

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary_path = Path(RESULT_DIR) / 'preprocessing_summary.json'
manifest_path = Path(RESULT_DIR) / 'preprocessing_manifest.csv'
failures_path = Path(RESULT_DIR) / 'preprocessing_failures.csv'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\n날짜별 상태:')
display(pd.read_csv(manifest_path).tail(20))
failures = pd.read_csv(failures_path)
print('실패/미완성 날짜:', len(failures))
if len(failures):
    display(failures.head(20))